# Ensemble

Ranges of outcomes across an ensemble recipe's members ([ADR-006](../docs/architecture/ADR-006-ensemble-scenarios.md)): pick a recipe generated and run via `sim run <recipe> --ensemble-size N`, see its members' net worth (or any one account's balance) as a percentile-band ("fan") chart, plus a year-end table of mean/median/SD, the 5th-95th percentiles and the tail means -- the same numbers `sim ensemble summary <recipe>` prints. Then compare recipes on risk, not just their member 0 -- the same numbers `sim ensemble compare <baseline> <variant>` prints.

An ensemble recipe is an ordinary scenario whose `calendar.csv` has one or more rows with `vary_distribution` set; `sim run` detects this and generates `state/runs/<recipe>/member_NNNN.sqlite3` per member instead of a single run file. Recipes with no members yet don't appear in the dropdowns below.

> **Note:** after running cells, `git status` will show this notebook as modified even with no real changes. Staging it (`git add`) will make it show unmodified again, unless actual changes are present -- see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it) for why.

In [ ]:
%load_ext autoreload
%autoreload 2

from analysis.sim.notebook_ui.ensemble_widgets import ensemble_compare_widget, ensemble_widget


## Browse an ensemble

- **Recipe**: an ensemble scenario with at least one generated member under `RUNS_DIR`. Re-run the cell below after generating a new one.
- **Account**: net worth, or any one account's balance.
- **Reserve**: CPI-constant dollars. Blank uses the recipe's `scenario.csv` `meta_json` `reserve`, which applies to net worth only; a typed value applies to whatever is shown.
- **Chart**: every month's mean and median across members, over quintile bands shaded darker toward the middle and out to the 5th-95th percentiles, with the reserve as a dashed line in CPI-constant dollars.
- **Table**: mean, median, SD, the `p05`, `p20`-`p80`, `p95` percentiles, and `cvar05`/`cvar20`, the mean of the worst 5% and 20% of members, one row per year-end. With a reserve, `below_reserve` is the share of members below it at that year end and `ever_below_reserve` the share that have been below it at any month so far. No min or max: with many members they're one member's extreme luck.

In [ ]:
ensemble_widget(show_description=True)

## Compare ensembles on risk

Each variant against the baseline, for net worth or one account:

- **Percentile deltas**: each breakpoint minus the same breakpoint of the baseline. Δp05 above zero and Δp95 below it means the variant raises the floor and gives up the top. The members at a breakpoint are different futures on each side.
- **Future by future**: when the two recipes share a seed (a child inherits its parent's), member *i* of each drew the same future for every row they share, so `variant − baseline` member by member is a real per-future difference. `P(ahead)` in the table is the share of those futures where the variant comes out ahead.
- **CVaR deltas**: the change in the mean of the worst 5% and 20% of members. A fixed, standard pair: steadier than one breakpoint, and it says how bad the bad futures are.
- **Tail and reserve**: each recipe's tail means and reserve shares, by year end.

With few members the low percentiles are noisy; rerun with more members (about 80 for a quick look, 400 to validate) rather than reading small differences.

In [ ]:
ensemble_compare_widget()